# Backbone / Architecture / Decoder Comparison

This notebook loads result CSVs and compares performance by backbone, architecture, and encoder weights.

It parses the `Experiment` field in result files such as:
- `Unet-resnet34-imagenet_Single-TB`
- `DeepLabV3Plus-resnet50-imagenet_Shift-TA_TC-to-TB_10pct`


## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path('/users/7/yu001011/csci5527/CSCI5527-final')
RUNS_DIR = PROJECT_ROOT / 'baseline_models_scripts' / 'runs'
BASELINE_CSV = PROJECT_ROOT / 'baseline_models_scripts' / 'runs' / 'results_Unet_resnet34.csv'

PROJECT_ROOT, RUNS_DIR, BASELINE_CSV

(PosixPath('/users/7/yu001011/csci5527/CSCI5527-final'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/results_Unet_resnet34.csv'))

## 2. Collect Result Files

By default, we load all CSV files inside `baseline_models_scripts/runs/` and the baseline CSV.
Add or remove paths if needed.

In [3]:
csv_paths = sorted(RUNS_DIR.glob('*.csv'))
if BASELINE_CSV.exists():
    csv_paths.append(BASELINE_CSV)

display(Markdown('Loaded CSV files:'))
for p in csv_paths:
    display(Markdown(f'- `{p}`'))

Loaded CSV files:

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer2_efficientnet-b0.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer2_resnet34.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer2_resnet50.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer3_arch_DeepLabV3Plus.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer3_arch_FPN.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer3_arch_UnetPlusPlus.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/results_Unet_resnet34.csv`

- `/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/results_Unet_resnet34.csv`

## 3. Load and Merge Results

In [4]:
def load_results(paths):
    frames = []
    for path in paths:
        try:
            df = pd.read_csv(path)
            df['source_file'] = path.name
            frames.append(df)
        except Exception as exc:
            print(f'Skip {path}: {exc}')
    if not frames:
           return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

results_df = load_results(csv_paths)
results_df.head()

,Experiment,Val_Loss,Val_IoU,Val_F1,Val_Recall,Val_Prec,Test_Loss,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
0,Unet-efficientnet-b0-imagenet_Single-TB,5.843002,0.408674,0.573785,0.641504,0.558950,6.675372,0.269279,0.424301,0.390885,0.486734,layer2_efficientnet-b0.csv
1,Unet-efficientnet-b0-imagenet_Shift-TA_TC-to-T...,5.620159,0.502825,0.644656,0.702345,0.617899,6.485694,0.370003,0.534739,0.678412,0.449778,layer2_efficientnet-b0.csv
2,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,5.420809,0.579151,0.725868,0.777058,0.697560,5.524915,0.377296,0.545845,0.586488,0.510535,layer2_efficientnet-b0.csv
3,UnetPlusPlus-resnet34-imagenet_Single-TB,5.708098,0.450820,0.612039,0.706195,0.566607,6.303780,0.319874,0.484400,0.433336,0.555699,layer2_resnet34.csv
4,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,5.604788,0.565174,0.701180,0.742216,0.684565,6.530971,0.308278,0.470939,0.571804,0.419581,layer2_resnet34.csv


## 4. Parse Experiment Name

Expected pattern:
`<arch>-<encoder>-<weights>_<setting>`

In [5]:
def parse_experiment(exp_name: str):
    if '_' not in exp_name:
        return {'arch': None, 'encoder': None, 'weights': None, 'setting': exp_name}

    model_part, setting = exp_name.split('_', 1)
    parts = model_part.split('-')

    if len(parts) < 3:
        return {'arch': parts[0], 'encoder': None, 'weights': None, 'setting': setting}

    arch = parts[0]
    weights = parts[-1]
    encoder = '-'.join(parts[1:-1])

    return {
        'arch': arch,
        'encoder': encoder,
        'weights': weights,
        'setting': setting,
    }

parsed = results_df['Experiment'].apply(parse_experiment).apply(pd.Series)
results_df = pd.concat([results_df, parsed], axis=1)
results_df[['Experiment', 'arch', 'encoder', 'weights', 'setting']].head()

,Experiment,arch,encoder,weights,setting
0,Unet-efficientnet-b0-imagenet_Single-TB,Unet,efficientnet-b0,imagenet,Single-TB
1,Unet-efficientnet-b0-imagenet_Shift-TA_TC-to-T...,Unet,efficientnet-b0,imagenet,Shift-TA_TC-to-TB_10pct
2,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,Unet,efficientnet-b0,imagenet,Shift-TA_TB-to-TC_10pct
3,UnetPlusPlus-resnet34-imagenet_Single-TB,UnetPlusPlus,resnet34,imagenet,Single-TB
4,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,UnetPlusPlus,resnet34,imagenet,Shift-TA_TC-to-TB_10pct


## 5. Quick Filters

Edit the lists to focus on specific backbones, architectures, or settings.

In [8]:
FOCUS_SETTINGS = [
    'Single-TB',
    'Shift-TA_TC-to-TB_10pct',
    # 'Shift-TA_TB-to-TC_10pct',
]

FOCUS_ARCH = ['Unet', 'DeepLabV3Plus', 'UnetPlusPlus', 'FPN', 'DeepLabV3Plus']  # e.g. ['Unet', 'DeepLabV3Plus']
FOCUS_ENCODERS = ['resnet34']  # e.g. ['resnet34', 'efficientnet-b0', 'resnet50']
FOCUS_WEIGHTS = []  # e.g. ['imagenet']

METRICS = ['Test_IoU', 'Test_F1', 'Test_Recall', 'Test_Prec']

filtered = results_df.copy()
if FOCUS_SETTINGS:
    filtered = filtered[filtered['setting'].isin(FOCUS_SETTINGS)]
if FOCUS_ARCH:
    filtered = filtered[filtered['arch'].isin(FOCUS_ARCH)]
if FOCUS_ENCODERS:
    filtered = filtered[filtered['encoder'].isin(FOCUS_ENCODERS)]
if FOCUS_WEIGHTS:
    filtered = filtered[filtered['weights'].isin(FOCUS_WEIGHTS)]

display_cols = ['Experiment', 'arch', 'encoder', 'weights', 'setting'] + METRICS
display(filtered[display_cols])

,Experiment,arch,encoder,weights,setting,Test_IoU,Test_F1,Test_Recall,Test_Prec
3,UnetPlusPlus-resnet34-imagenet_Single-TB,UnetPlusPlus,resnet34,imagenet,Single-TB,0.319874,0.484400,0.433336,0.555699
4,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,UnetPlusPlus,resnet34,imagenet,Shift-TA_TC-to-TB_10pct,0.308278,0.470939,0.571804,0.419581
8,DeepLabV3Plus-resnet34-imagenet_Single-TB,DeepLabV3Plus,resnet34,imagenet,Single-TB,0.307359,0.464434,0.488746,0.446283
9,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TC-to...,DeepLabV3Plus,resnet34,imagenet,Shift-TA_TC-to-TB_10pct,0.260046,0.406658,0.586841,0.320602
10,FPN-resnet34-imagenet_Single-TB,FPN,resnet34,imagenet,Single-TB,0.245093,0.393045,0.390756,0.404728
11,FPN-resnet34-imagenet_Shift-TA_TC-to-TB_10pct,FPN,resnet34,imagenet,Shift-TA_TC-to-TB_10pct,0.236131,0.378628,0.572779,0.291442
12,UnetPlusPlus-resnet34-imagenet_Single-TB,UnetPlusPlus,resnet34,imagenet,Single-TB,0.319874,0.484400,0.433336,0.555699
13,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,UnetPlusPlus,resnet34,imagenet,Shift-TA_TC-to-TB_10pct,0.308278,0.470939,0.571804,0.419581
15,Unet-resnet34-imagenet_Single-TB,Unet,resnet34,imagenet,Single-TB,0.356572,0.502834,0.559124,0.468969
20,Unet-resnet34-imagenet_Shift-TA_TC-to-TB_10pct,Unet,resnet34,imagenet,Shift-TA_TC-to-TB_10pct,0.295580,0.455053,0.617487,0.383615


## 6. Backbone Comparison (within a fixed architecture)

This pivot shows `Test_IoU` by backbone for each setting.

In [9]:
if not filtered.empty:
    for metric in METRICS:
        pivot_backbone = filtered.pivot_table(
            index='setting',
            columns='encoder',
            values=metric,
            aggfunc='mean'
        )
        display(Markdown(f'### {metric} by Backbone'))
        display(pivot_backbone)
else:
    display(Markdown('No data after filtering.'))

### Test_IoU by Backbone

encoder,resnet34
setting,
Shift-TA_TC-to-TB_10pct,0.283982
Single-TB,0.317557


### Test_F1 by Backbone

encoder,resnet34
setting,
Shift-TA_TC-to-TB_10pct,0.439545
Single-TB,0.471991


### Test_Recall by Backbone

encoder,resnet34
setting,
Shift-TA_TC-to-TB_10pct,0.589700
Single-TB,0.477404


### Test_Prec by Backbone

encoder,resnet34
setting,
Shift-TA_TC-to-TB_10pct,0.369740
Single-TB,0.483391


## 7. Architecture Comparison (within a fixed backbone)

This pivot shows `Test_IoU` by architecture for each setting.

In [10]:
if not filtered.empty:
    for metric in METRICS:
        pivot_arch = filtered.pivot_table(
            index='setting',
            columns='arch',
            values=metric,
            aggfunc='mean'
        )
        display(Markdown(f'### {metric} by Architecture'))
        display(pivot_arch)
else:
    display(Markdown('No data after filtering.'))

### Test_IoU by Architecture

arch,DeepLabV3Plus,FPN,Unet,UnetPlusPlus
setting,,,,
Shift-TA_TC-to-TB_10pct,0.260046,0.236131,0.295580,0.308278
Single-TB,0.307359,0.245093,0.356572,0.319874


### Test_F1 by Architecture

arch,DeepLabV3Plus,FPN,Unet,UnetPlusPlus
setting,,,,
Shift-TA_TC-to-TB_10pct,0.406658,0.378628,0.455053,0.470939
Single-TB,0.464434,0.393045,0.502834,0.484400


### Test_Recall by Architecture

arch,DeepLabV3Plus,FPN,Unet,UnetPlusPlus
setting,,,,
Shift-TA_TC-to-TB_10pct,0.586841,0.572779,0.617487,0.571804
Single-TB,0.488746,0.390756,0.559124,0.433336


### Test_Prec by Architecture

arch,DeepLabV3Plus,FPN,Unet,UnetPlusPlus
setting,,,,
Shift-TA_TC-to-TB_10pct,0.320602,0.291442,0.383615,0.419581
Single-TB,0.446283,0.404728,0.468969,0.555699


## 8. Decoder / Architecture + Backbone Joint View

This table helps you compare `(arch, encoder)` pairs directly.

In [65]:
if not filtered.empty:
    filtered = filtered.copy()
    filtered['arch+encoder'] = filtered['arch'].fillna('') + '+' + filtered['encoder'].fillna('')
    for metric in METRICS:
        pivot_pair = filtered.pivot_table(
            index='setting',
            columns='arch+encoder',
            values=metric,
            aggfunc='mean'
        )
        display(Markdown(f'### {metric} by (Architecture + Backbone)'))
        display(pivot_pair)
else:
    display(Markdown('No data after filtering.'))

### Test_IoU by (Architecture + Backbone)

arch+encoder,Unet+efficientnet-b0,Unet+resnet34,Unet+resnet50
setting,,,
Shift-TA_TB-to-TC_10pct,0.377296,0.345064,0.264646
Shift-TA_TC-to-TB_10pct,0.370003,0.295580,0.305875
Single-TB,0.269279,0.356572,0.335794


### Test_F1 by (Architecture + Backbone)

arch+encoder,Unet+efficientnet-b0,Unet+resnet34,Unet+resnet50
setting,,,
Shift-TA_TB-to-TC_10pct,0.545845,0.512940,0.416388
Shift-TA_TC-to-TB_10pct,0.534739,0.455053,0.467060
Single-TB,0.424301,0.502834,0.492554


### Test_Recall by (Architecture + Backbone)

arch+encoder,Unet+efficientnet-b0,Unet+resnet34,Unet+resnet50
setting,,,
Shift-TA_TB-to-TC_10pct,0.586488,0.580492,0.433505
Shift-TA_TC-to-TB_10pct,0.678412,0.617487,0.555109
Single-TB,0.390885,0.559124,0.497126


### Test_Prec by (Architecture + Backbone)

arch+encoder,Unet+efficientnet-b0,Unet+resnet34,Unet+resnet50
setting,,,
Shift-TA_TB-to-TC_10pct,0.510535,0.462844,0.424765
Shift-TA_TC-to-TB_10pct,0.449778,0.383615,0.414485
Single-TB,0.486734,0.468969,0.496959


## 9. Quick Summary

Use these prompts to write conclusions:

- Which backbone improves `Test_IoU` the most on the hard settings?
- Do architecture changes help more than backbone changes?
- Are gains consistent across settings, or only in one target domain?
